# EJERCICIOS

### 9. Intenta modificar el modelo Seq2SeqModel para pronosticar tanto el tráfico de tren como el de autobús para los próximos 14 días. El modelo ahora necesitará predecir 28 valores en lugar de 14.

In [1]:
import torch.nn as nn
# Implementando en base a principios RNN
class SimpleRnnModel(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.hidden_size = hidden_size
        self.memory_cell = nn.Sequential(
        nn.Linear(input_size + hidden_size, hidden_size),
        nn.Tanh()
        )
        
        self.output = nn.Linear(hidden_size, output_size)

    def forward(self, X):
        
        batch_size, seq_len, _ = X.shape
        h = torch.zeros(batch_size, self.hidden_size, device=X.device, dtype=X.dtype)
    
        for t in range(seq_len):
            x_t = X[:, t, :]                       # (batch, input_size)
            combined = torch.cat([x_t, h], dim=1)  # (batch, input_size + hidden_size)
            h = self.memory_cell(combined)         # (batch, hidden_size)
    
        # Solo devolvemos la predicción a partir del último hidden state
        # (este modelo predice "de un solo disparo" al final de la ventana,
        # no en cada timestep)
        return self.output(h)                       # (batch, output_size)

In [2]:
import torch
class TimeSeriesDataset(torch.utils.data.Dataset):
    def __init__(self, series, window_length):
        self.series = series
        self.window_length = window_length

    def __len__(self):
        return len(self.series) - self.window_length

    def __getitem__(self, idx):
        if idx >= len(self):
            raise IndexError("dataset index out of range")
        
        end = idx + self.window_length # 1st index after window
        window = self.series[idx : end]
        target = self.series[end]
        
        return window, target

class ForecastAheadDataset(TimeSeriesDataset):
    def __len__(self):
        #
        return len(self.series) - self.window_length - 14 + 1

    def __getitem__(self, idx):
        end = idx + self.window_length # 1st index after window
        window = self.series[idx : end]
        
        target = self.series[end : end + 14, 0] # 0 = rail ridership
        return window, target

class Seq2SeqDatasetMultivar(ForecastAheadDataset):
    def __getitem__(self, idx):
        end = idx + self.window_length # 1st index after window
        window = self.series[idx : end]

        target_rail = self.series[end : end + 14, 0] # (14,)
        target_bus  = self.series[end : end + 14, 1] # (14,)
        target = torch.cat([target_rail, target_bus], dim=0) # (28,)

        return window, target

In [3]:
import pandas as pd
from pathlib import Path
from torch.utils.data import DataLoader

window_length = 56

""" Cargamos el csv, cambiamos el nombre de las columnas, ordenamos por
fecha, eliminamos información redundante """
path = Path("CTA_-_Ridership_-_Daily_Boarding_Totals_20250830.csv")
df = pd.read_csv(path, parse_dates=["service_date"], thousands=",")
df.columns = ["date", "day_type", "bus", "rail", "total"] # shorter names
df = df.sort_values("date").set_index("date")
df = df.drop("total", axis=1) # no need for total, it's just bus + rail
df = df.drop_duplicates() # remove duplicated months (2011-10 and 2014-07)


# Crearmos un df multivariable escalado
df_mulvar = df[["rail", "bus"]] / 1e6 
# Definimos el dia siguiente para tenerlo en cuenta en el entrenamiento
df_mulvar["next_day_type"] = df["day_type"].shift(-1) 
# Codificamos el tipo de dia a float
df_mulvar = pd.get_dummies(df_mulvar, dtype=float) 

mulvar_train = torch.FloatTensor(df_mulvar["2016-01":"2018-12"].values )
mulvar_valid = torch.FloatTensor(df_mulvar["2019-01":"2019-05"].values )
mulvar_test = torch.FloatTensor(df_mulvar["2019-06":].values )

seq2seq_train_set = Seq2SeqDatasetMultivar(mulvar_train, window_length)
seq2seq_train_loader = DataLoader(seq2seq_train_set, batch_size=32, shuffle=True)
seq2seq_eval_set = Seq2SeqDatasetMultivar(mulvar_valid, window_length)
seq2seq_eval_loader = DataLoader(seq2seq_eval_set, batch_size=32, shuffle=True)
seq2seq_test_set = Seq2SeqDatasetMultivar(mulvar_test, window_length)
seq2seq_test_loader = DataLoader(seq2seq_test_set, batch_size=32, shuffle=True)


In [4]:

import copy

def entrenar_cnn(modelo: nn.Module, optimizador: torch.optim, perdida, metrica, entrenamiento, validacion, n_iter, device, scheduler = None, delta = 0.001, paciencia = 10):
    """ Función de entrenamiento para los modelos convolucionales predictivos  """

    mejor_precision = float("inf")
    mejor_modelo = copy.deepcopy(modelo.state_dict())
    contador_paciencia = 0

    for iteracion in range(n_iter):
        #--------------------- ENTRENAMIENTO --------------------- 
        modelo.train()

        perdida_total = 0.0
        perdida_eval_total = 0.0
        metrica.reset()
        for batch in entrenamiento:
            # Los DataLoader con secuencias de longitud variable (los que usan
            # el collate_fn con padding) devuelven un tercer elemento con la
            # longitud real de cada secuencia del batch; los datasets de
            # ventana fija (ejercicios 9 y 10) siguen devolviendo solo
            # (muestra, etiqueta), así que aquí soportamos ambos casos
            if len(batch) == 3:
                muestra, etiqueta, longitudes = batch
            else:
                muestra, etiqueta = batch
                longitudes = None

            muestra = muestra.to(device)
            etiqueta = etiqueta.to(device)


            optimizador.zero_grad()

            pred = modelo(muestra, longitudes) if longitudes is not None else modelo(muestra)

            perdida_entrenamiento = perdida(pred, etiqueta)
            perdida_total += perdida_entrenamiento.item()

            perdida_entrenamiento.backward()
            optimizador.step()
            metrica.update(pred, etiqueta)
            metrica_entrenamiento = metrica.compute().item()
        
        precision_eval = eval_entrenamiento(modelo, validacion, metrica, device)
        if scheduler:
            if isinstance(scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):
                scheduler.step(precision_eval.item())

            else:
                scheduler.step()

        

        print(f"Epoca {iteracion} / {n_iter}: \nPerdida entrenamiento {perdida_total / len(entrenamiento):.4f} Metrica entrenamiento {metrica_entrenamiento:.4f} \nMetrica validacion {precision_eval.item():.4f}")

        # FIX: antes era "precision_eval.item() > mejor_precision + delta"
        if precision_eval.item() < mejor_precision - delta:
            # Mejora significativa
            mejor_precision = precision_eval.item()
            mejor_modelo = copy.deepcopy(modelo.state_dict())
            contador_paciencia = 0
        else:
            # Sin mejora, o mejora demasiado pequeña para contar
            contador_paciencia += 1
            if contador_paciencia >= paciencia:
                print(f"Early stopping activado en la epoca {iteracion + 1}. Mejor precisión {mejor_precision:.4f}")
                break
    print(f"Entrenamiento acabado. Mejor precisión {mejor_precision:.4f}")
    return mejor_modelo

def eval_entrenamiento(modelo, validacion, metrica, device):
    #--------------------- EVALUACION --------------------- 
    modelo.eval()
    metrica.reset()
    with torch.no_grad():
        for batch in validacion:
            if len(batch) == 3:
                muestra, etiqueta, longitudes = batch
            else:
                muestra, etiqueta = batch
                longitudes = None

            muestra = muestra.to(device)
            etiqueta = etiqueta.to(device)

            pred = modelo(muestra, longitudes) if longitudes is not None else modelo(muestra)
            metrica.update(pred, etiqueta)

    return metrica.compute()
                  

In [5]:
import torchmetrics
import torch
torch.manual_seed(42)
ahead_model = SimpleRnnModel(input_size=5, hidden_size=32, output_size=28)
ahead_model = ahead_model.to("cuda")

mae = nn.MSELoss().to("cuda")
optimizer = torch.optim.Adam(ahead_model.parameters(), lr=1e-3, weight_decay=1e-4)
metrica = torchmetrics.MeanAbsoluteError().to("cuda")

model = entrenar_cnn(ahead_model, optimizer, mae, metrica, seq2seq_train_loader, seq2seq_eval_loader, 40, "cuda", scheduler = None, delta = 0.001, paciencia = 10)


Epoca 0 / 40: 
Perdida entrenamiento 0.3028 Metrica entrenamiento 0.4979 
Metrica validacion 0.2421
Epoca 1 / 40: 
Perdida entrenamiento 0.0496 Metrica entrenamiento 0.1821 
Metrica validacion 0.1426
Epoca 2 / 40: 
Perdida entrenamiento 0.0328 Metrica entrenamiento 0.1574 
Metrica validacion 0.1418
Epoca 3 / 40: 
Perdida entrenamiento 0.0306 Metrica entrenamiento 0.1528 
Metrica validacion 0.1372
Epoca 4 / 40: 
Perdida entrenamiento 0.0279 Metrica entrenamiento 0.1454 
Metrica validacion 0.1275
Epoca 5 / 40: 
Perdida entrenamiento 0.0238 Metrica entrenamiento 0.1307 
Metrica validacion 0.1094
Epoca 6 / 40: 
Perdida entrenamiento 0.0185 Metrica entrenamiento 0.1089 
Metrica validacion 0.0933
Epoca 7 / 40: 
Perdida entrenamiento 0.0152 Metrica entrenamiento 0.0955 
Metrica validacion 0.0853
Epoca 8 / 40: 
Perdida entrenamiento 0.0137 Metrica entrenamiento 0.0893 
Metrica validacion 0.0776
Epoca 9 / 40: 
Perdida entrenamiento 0.0125 Metrica entrenamiento 0.0832 
Metrica validacion 0.0696


### 10. Descarga el conjunto de datos de corales de Bach y descomprímelo. Está compuesto por 382 corales compuestos por Johann Sebastian Bach. Cada coral tiene entre 100 y 640 pasos de tiempo de longitud, y cada paso de tiempo contiene 4 enteros, donde cada entero corresponde al índice de una nota en un piano (excepto el valor 0, que significa que no se toca ninguna nota). Entrena un modelo —recurrente, convolucional, o ambos— que pueda predecir el siguiente paso de tiempo (cuatro notas), dada una secuencia de pasos de tiempo de un coral. Luego usa este modelo para generar música al estilo de Bach, una nota a la vez: puedes hacerlo dando al modelo el comienzo de un coral y pidiéndole que prediga el siguiente paso de tiempo, luego añadiendo estos pasos de tiempo a la secuencia de entrada y pidiendo al modelo la siguiente nota, y así sucesivamente. Asegúrate también de revisar el modelo Coconet de Google, que se usó para un bonito doodle de Google sobre Bach.

In [6]:
import tarfile
import urllib.request
from pathlib import Path

# Mismo repositorio de datasets que usa Géron para el resto de ejercicios del libro
url = "https://github.com/ageron/data/raw/main/jsb_chorales.tgz"
tgz_path = Path("jsb_chorales.tgz")
bach_path = Path("jsb_chorales")

if not bach_path.exists():
    if not tgz_path.exists():
        urllib.request.urlretrieve(url, tgz_path)

    with tarfile.open(tgz_path) as tar:
        tar.extractall(path=".")

# Cada corale es un CSV con columnas note0..note3 (un entero = índice de nota
# de piano en cada paso de tiempo; 0 = silencio)
train_files = sorted((bach_path / "train").glob("chorale_*.csv"))
valid_files = sorted((bach_path / "valid").glob("chorale_*.csv"))
test_files  = sorted((bach_path / "test").glob("chorale_*.csv"))

print(f"train: {len(train_files)}  valid: {len(valid_files)}  test: {len(test_files)}")
print(f"total: {len(train_files) + len(valid_files) + len(test_files)}")

train: 229  valid: 76  test: 77
total: 382


##### Modelo secuencia a secuencia que reciba n pasos de tiempo (4 notas) y sea capaz de predecir 1 paso de tiempo (4 notas). Idea principal: - Implementar un modelo con capas convolucionales que reduzcan la dimensión de la secuencia del coral + GRU como célula de memória + capa linear para la salida. - Implementar un dataset para montar los lotes de archivos, leer el archivo, guardar el coral y devolver una ventana con n pasos de tiempo por iteración      
Formato csv:  
temp_step note0  note1  note2  note3  
0       74     70     65     58

In [7]:
import torch.nn.functional as F
class CausalConv1d(nn.Conv1d):
   
    def forward(self, X):
        padding = (self.kernel_size[0] - 1) * self.dilation[0]
        X = F.pad(X, (padding, 0))
        return super().forward(X)

def use_xavier_init(module):
    if isinstance(module, nn.Conv1d):
        nn.init.xavier_uniform_(module.weight)
        nn.init.zeros_(module.bias)
        
class GruModel(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
       
        super().__init__()
        self.hidden_size = hidden_size
        capas_conv = []
        for dilatacion in (1,2,4) *2:
            conv = CausalConv1d(input_size, hidden_size, kernel_size=2, dilation=dilatacion)
            capas_conv += [conv, nn.ELU(alpha=0.2)]
            input_size = hidden_size
        
        self.conv_net = nn.Sequential(*capas_conv)
        use_xavier_init(self.conv_net)
        
        self.memory_cell = nn.GRUCell(input_size, hidden_size)
        self.output = nn.Linear(hidden_size, output_size)

    def forward(self, X):
        Z = X.permute(0, 2, 1)
        seq_colapsada = self.conv_net(Z)
        # [batch_size, hidden_size, len_seq]
        # Para iterar paso a paso con la GRU necesitamos tiempo primero
        seq_conv = seq_colapsada.permute(2, 0, 1)
        batch_size= X.shape[0]
        H = torch.zeros(batch_size, self.hidden_size, device=X.device)
        for X_t in seq_conv:
            # La celula GRU combina X_t con el H anterior y devuelve
            # directamente el H actualizado, ya con las puertas de
            # actualizacion y reinicio aplicadas por debajo
            H = self.memory_cell(X_t, H)
        # Tras el bucle, H es el estado final, que aqui hace a la vez de
        # estado a corto y a largo plazo; se pasa por la capa de salida para
        # obtener la prediccion, de forma [batch_size, output_size]
        return self.output(H)
    

In [8]:
import pandas as pd
"""
Formato csv:  
temp_step note0  note1  note2  note3  
0       74     70     65     58
"""
class CoralDataset(torch.utils.data.Dataset):
    def __init__(self, series, window_length, transforms = None):
        self.series = series
        self.window_length = window_length
        self.transforms = transforms

    def __len__(self):
        return len(self.series) - self.window_length

    def __getitem__(self, idx):
        end = idx + self.window_length
        if idx >= len(self):
            raise IndexError("dataset index out of range")
        
        window = self.series[idx : end]
        target = self.series[end]

        if self.transforms:
            window = self.transforms(window)
            target = self.transforms(target)
        
        return window, target
import torchvision.transforms.v2 as T

def montar_dt_x_coral(lista):
    lista_dt = list()
    for coral in lista:
        # No se puede pasar el path al csv directamente, tienes que leer el archivo y pasar los valores al dataset
        csv = pd.read_csv(coral)
        
        tensor = torch.tensor(csv.values,dtype=torch.float32)
        lista_dt.append(CoralDataset(tensor, 20))
    return torch.utils.data.ConcatDataset(lista_dt)

train_dt = montar_dt_x_coral(train_files)
train_loader = DataLoader(train_dt, batch_size=32, shuffle=True)

eval_dt = montar_dt_x_coral(valid_files)
eval_loader = DataLoader(eval_dt, batch_size=32, shuffle=True)

test_dt = montar_dt_x_coral(test_files)
test_loader = DataLoader(test_dt, batch_size=32, shuffle=True)


In [9]:
torch.manual_seed(42)
conv_gru_model = GruModel(input_size=4, hidden_size=32, output_size=4)
conv_gru_model = conv_gru_model.to("cuda")


mae = nn.MSELoss().to("cuda")
optimizer2 = torch.optim.AdamW(conv_gru_model.parameters(), betas=(0.9, 0.999) , lr=0.002, weight_decay=1e-4)
metrica = torchmetrics.MeanAbsoluteError().to("cuda")
adapt_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer2, mode="min", patience=3, factor=0.65)
model = entrenar_cnn(conv_gru_model, optimizer2, mae, metrica, train_loader, eval_loader, 40, "cuda", scheduler = adapt_scheduler, delta = 0.1, paciencia = 7)


Epoca 0 / 40: 
Perdida entrenamiento 883.6394 Metrica entrenamiento 23.8776 
Metrica validacion 4.7557
Epoca 1 / 40: 
Perdida entrenamiento 27.5720 Metrica entrenamiento 3.8690 
Metrica validacion 3.5189
Epoca 2 / 40: 
Perdida entrenamiento 22.2315 Metrica entrenamiento 3.3474 
Metrica validacion 3.3856
Epoca 3 / 40: 
Perdida entrenamiento 20.5961 Metrica entrenamiento 3.1320 
Metrica validacion 3.3859
Epoca 4 / 40: 
Perdida entrenamiento 20.6062 Metrica entrenamiento 3.1328 
Metrica validacion 3.3888
Epoca 5 / 40: 
Perdida entrenamiento 20.3742 Metrica entrenamiento 3.1564 
Metrica validacion 3.3398
Epoca 6 / 40: 
Perdida entrenamiento 19.8030 Metrica entrenamiento 3.1329 
Metrica validacion 3.3281
Epoca 7 / 40: 
Perdida entrenamiento 19.7297 Metrica entrenamiento 3.1273 
Metrica validacion 3.3301
Epoca 8 / 40: 
Perdida entrenamiento 19.7216 Metrica entrenamiento 3.1276 
Metrica validacion 3.3287
Epoca 9 / 40: 
Perdida entrenamiento 19.7093 Metrica entrenamiento 3.1272 
Metrica valida

In [10]:
muestra,etiqueta = next(iter(test_loader))
muestra = muestra[:1].to("cuda")
gen_coral = []
with torch.no_grad():
    for i in range(0,40):
        pred = conv_gru_model(muestra)
        gen_coral.append(pred)


## 11. Entrena un modelo de clasificación para el conjunto de datos QuickDraw, que contiene millones de bocetos de varios objetos. Empieza descargando los datos simplificados de unas pocas clases (por ejemplo, ant.ndjson, axe.ndjson y bat.ndjson). Cada archivo NDJSON contiene un objeto JSON por línea, que puedes analizar usando la función json.loads() de Python. Esto te dará una lista de bocetos, donde cada boceto se representa como un diccionario de Python. En cada diccionario, la entrada "drawing" contiene una lista de trazos del lápiz. Puedes convertir esta lista en un tensor 3D de flotantes donde las dimensiones son [trazos, coordenadas x, coordenadas y]. Dado que una RNN toma una única secuencia como entrada, necesitarás concatenar todos los trazos de cada boceto en una única secuencia. Es mejor añadir una característica extra para que la RNN sepa cuánto ha avanzado en cada trazo actualmente (por ejemplo, de 0 a 1). En otras palabras, el modelo recibirá una secuencia donde cada paso de tiempo tiene tres características: las coordenadas x e y del lápiz, y la proporción de progreso a lo largo del trazo actual.

In [11]:
import urllib.request
from pathlib import Path
import json
import pandas as pd
# Datos simplificados de QuickDraw (un objeto JSON por linea, formato ndjson)
base_url = "https://storage.googleapis.com/quickdraw_dataset/full/simplified/"
quickdraw_path = Path("quickdraw")
quickdraw_path.mkdir(exist_ok=True)

# Empezamos con unas pocas clases, tal y como sugiere el enunciado
clases = ["ant", "axe", "bat"]

for clase in clases:
    archivo = quickdraw_path / f"{clase}.ndjson"
    if not archivo.exists():
        url = base_url + f"{clase}.ndjson"
        print(f"Descargando {clase}...")
        urllib.request.urlretrieve(url, archivo)
lista_clases = []
for clase in clases:
    archivo = quickdraw_path / f"{clase}.ndjson"
    print(f"{clase}: {archivo} {archivo.stat().st_size / 1e6:.1f} MB")
    df = pd.read_json(archivo, lines=True)
    
    df = df.drop("timestamp", axis='columns')
    df = df.drop("recognized", axis='columns')
    df = df.drop("countrycode", axis='columns')
    df = df.drop("key_id", axis='columns')
    lista_clases.append(df)


ant: quickdraw/ant.ndjson 72.3 MB
axe: quickdraw/axe.ndjson 41.0 MB
bat: quickdraw/bat.ndjson 60.8 MB


In [12]:
lista_tensores = []
for clase in lista_clases:
    tensores = []
    for boceto in clase["drawing"]:
        puntos = []
        for x_coords, y_coords in boceto:
            n = len(x_coords)
            for i in range(n):
                progreso = i / (n - 1) if n > 1 else 0.0
                puntos.append([x_coords[i], y_coords[i], progreso])
        tensores.append(torch.tensor(puntos, dtype=torch.float32))
    lista_tensores.append(tensores)


In [13]:
"""
Formato tensor:  
trazo coord_x  coord_y  progreso    
"""
class DibujosDataset(torch.utils.data.Dataset):
    def __init__(self, series, target):
        self.series = series
        self.target = target

    def __len__(self):
        return len(self.series)

    def __getitem__(self, idx):
        if idx >= len(self):
            raise IndexError("dataset index out of range")
        
        window = self.series[idx]
        target = self.target
        
        return window, target

# FIX: usamos el indice de la clase (0, 1, 2...) como target en vez del
# nombre en texto ("ant", "axe", "bat"). CrossEntropyLoss y
# torchmetrics.Accuracy necesitan enteros, y ademas collate_fn no puede
# convertir strings a tensor con torch.tensor()
ant_dt = DibujosDataset(lista_tensores[0], clases.index("ant"))
axe_dt = DibujosDataset(lista_tensores[1], clases.index("axe"))
bat_dt = DibujosDataset(lista_tensores[2], clases.index("bat"))

conc_dt = torch.utils.data.ConcatDataset([ant_dt, axe_dt, bat_dt])

n = len(conc_dt)
n_train = int(0.7 * n)
n_val = int(0.15 * n)
n_test = n - n_train - n_val

train_ds, eval_ds, test_ds = torch.utils.data.random_split(
    conc_dt, [n_train, n_val, n_test],
    generator=torch.Generator().manual_seed(42)
)

from torch.nn.utils.rnn import pad_sequence
# Función para definir dentro del dataloader como hacer padding en los tensores, debido a que
# en este caso son de longitud irregular. Cuando ejecuta torch.stark al crear los batches el codigo
# revienta por que no tienen la misma forma todos
def collate_fn(batch):
    windows, targets = zip(*batch)
    lengths = torch.tensor([w.shape[0] for w in windows])
    windows_padded = pad_sequence(windows, batch_first=True, padding_value=0.0)
    targets = torch.stack(targets) if torch.is_tensor(targets[0]) else torch.tensor(targets)
    return windows_padded, targets, lengths
    
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, collate_fn = collate_fn)

eval_loader = DataLoader(eval_ds, batch_size=32, shuffle=True, collate_fn = collate_fn)

test_loader = DataLoader(test_ds, batch_size=32, shuffle=True, collate_fn = collate_fn)


In [14]:
import torch.nn.functional as F
        
class GruModelPadded(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
       
        super().__init__()
        self.hidden_size = hidden_size
        capas_conv = []
        for dilatacion in (1,2,4) *2:
            conv = CausalConv1d(input_size, hidden_size, kernel_size=2, dilation=dilatacion)
            capas_conv += [conv, nn.ELU(alpha=0.2)]
            input_size = hidden_size
        
        self.conv_net = nn.Sequential(*capas_conv)
        use_xavier_init(self.conv_net)
        
        self.memory_cell = nn.GRUCell(input_size, hidden_size)
        self.output = nn.Linear(hidden_size, output_size)

    def forward(self, X, lengths):
        Z = X.permute(0, 2, 1)
        # La conv es causal (solo mira hacia atras), asi que el padding que
        # el DataLoader añade al final de las secuencias cortas no afecta a
        # los timesteps reales anteriores: es seguro convolucionar sobre el
        # batch ya rellenado tal cual
        seq_colapsada = self.conv_net(Z)
        # [batch_size, hidden_size, len_seq]
        # Para iterar paso a paso con la GRU necesitamos tiempo primero
        seq_conv = seq_colapsada.permute(2, 0, 1)
        batch_size= X.shape[0]
        H = torch.zeros(batch_size, self.hidden_size, device=X.device)

        # Este modelo usa GRUCell paso a paso (no nn.GRU), asi que no
        # podemos usar directamente pack_padded_sequence/pad_packed_sequence
        # (esas utilidades estan pensadas para los modulos nn.RNN/GRU/LSTM
        # que procesan la secuencia entera de una vez). Conseguimos el mismo
        # efecto "a mano" con una mascara por timestep
        lengths = lengths.to(X.device)

        for t, X_t in enumerate(seq_conv):
            # La celula GRU combina X_t con el H anterior y devuelve
            # directamente el H actualizado, ya con las puertas de
            # actualizacion y reinicio aplicadas por debajo
            H_nuevo = self.memory_cell(X_t, H)

            # mascara (batch_size, 1): True donde el timestep t todavia es
            # real para esa muestra (t < longitud real de su secuencia). Para
            # las muestras cuya secuencia ya termino, congelamos su H en vez
            # de dejar que el paso de la GRU lo actualice con el padding de
            # ceros, para que ese relleno no contamine el estado final que
            # se usa para predecir
            mascara = (t < lengths).unsqueeze(1)
            H = torch.where(mascara, H_nuevo, H)

        # Tras el bucle, H es el estado final de cada muestra en su ultimo
        # timestep real (no en el ultimo timestep con padding), que aqui hace
        # a la vez de estado a corto y a largo plazo; se pasa por la capa de
        # salida para obtener la prediccion, de forma [batch_size, output_size]
        return self.output(H)
    

In [15]:
torch.manual_seed(42)
# FIX: el modelo de clasificacion debe ser GruModelPadded (el que acepta
# `lengths` y soporta secuencias de longitud variable), y con output_size=3
# porque hay 3 clases (ant/axe/bat), no 1
class_model = GruModelPadded(input_size=3, hidden_size=32, output_size=3)
# FIX: antes esta linea sobreescribia class_model con conv_gru_model, que es
# el modelo entrenado en el ejercicio 10 (quedaba sin usar el modelo recien creado)
class_model = class_model.to("cuda")


xentropy = nn.CrossEntropyLoss().to("cuda")
optimizer3 = torch.optim.AdamW(class_model.parameters(), betas=(0.9, 0.999) , lr=0.002, weight_decay=1e-4)
accuracy = torchmetrics.Accuracy(num_classes=3, task="multiclass").to("cuda")
# mode="max" porque aqui la metrica es accuracy
# (mejor cuanto mas alta), no un error como el MAE del ejercicio 10 (mejor
# cuanto mas bajo)
adapt_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer3, mode="max", patience=3, factor=0.65)

model = entrenar_cnn(class_model, optimizer3, xentropy, accuracy, train_loader, eval_loader, 40, "cuda", scheduler = adapt_scheduler, delta = 0.1, paciencia = 7)


Epoca 0 / 40: 
Perdida entrenamiento 0.2269 Metrica entrenamiento 0.9169 
Metrica validacion 0.9429
Epoca 1 / 40: 
Perdida entrenamiento 0.1404 Metrica entrenamiento 0.9504 
Metrica validacion 0.9536
Epoca 2 / 40: 
Perdida entrenamiento 0.1286 Metrica entrenamiento 0.9548 
Metrica validacion 0.9590
Epoca 3 / 40: 
Perdida entrenamiento 0.1221 Metrica entrenamiento 0.9573 
Metrica validacion 0.9596
Epoca 4 / 40: 
Perdida entrenamiento 0.1201 Metrica entrenamiento 0.9580 
Metrica validacion 0.9565
Epoca 5 / 40: 
Perdida entrenamiento 0.1189 Metrica entrenamiento 0.9585 
Metrica validacion 0.9603
Epoca 6 / 40: 
Perdida entrenamiento 0.1176 Metrica entrenamiento 0.9587 
Metrica validacion 0.9616
Epoca 7 / 40: 
Perdida entrenamiento 0.1169 Metrica entrenamiento 0.9590 
Metrica validacion 0.9619
Early stopping activado en la epoca 8. Mejor precisión 0.9429
Entrenamiento acabado. Mejor precisión 0.9429


## 12. Crea un conjunto de datos que contenga grabaciones de audio cortas de ti diciendo "sí" o "no", y entrena una RNN de clasificación binaria con él. Por ejemplo, podrías:


### a. Usar un software de grabación de audio como Audacity para grabarte diciendo "sí" tantas veces como tu paciencia te lo permita, con pausas cortas entre cada palabra. Crea una grabación similar para la palabra "no". Intenta cubrir las diversas formas en que podrías pronunciar realmente estas palabras en la vida real.



### b. Cargar cada archivo WAV usando la función torchaudio.load() de la biblioteca TorchAudio. Esto devolverá un tensor que contiene el audio, así como un entero que indica el número de muestras por segundo. El tensor de audio tiene una forma de [canales, muestras]: un canal para mono, dos para estéreo. Convierte el estéreo a mono promediando sobre la dimensión de canal.



### c. Trocear cada grabación en palabras individuales dividiendo por los silencios. Puedes hacer esto usando la transformación torchaudio.transforms.Vad (Detección de Actividad de Voz).



### d. Dado que las secuencias son tan largas, es difícil entrenar directamente una RNN con ellas, así que ayuda convertir primero el audio en un espectrograma. Para esto, puedes usar la transformación torchaudio.transforms.MelSpectrogram, que es muy adecuada para la voz. La salida es una secuencia dramáticamente más corta, con muchos más canales.



### e. ¡Ahora intenta construir y entrenar una RNN de clasificación binaria con tu conjunto de datos de sí/no! Considera compartir tu conjunto de datos y modelo con el mundo (por ejemplo, a través de Hugging Face Hub).